# Exploratory Data Analysis: Day Approach

## Runner Injury Risk Prediction — Lovdal et al. (2021) Replication

This notebook explores the **day-level training load dataset** used to predict injury risk
in competitive runners. The dataset contains daily training snapshots from **74 elite Dutch
runners** spanning 2012–2019.

Each observation represents a **7-day sliding window** of 10 training metrics, creating a
70-feature representation of an athlete's recent training history. The target variable is
binary: **injury (1)** or **no injury (0)**.

### Key questions driving this analysis

1. **How severe is the class imbalance**, and does it vary across athletes?
2. **What do the training load distributions** reveal about the population?
3. **How does the sentinel value (−0.01)** encode rest days, and how prevalent is it?
4. **Which features show the strongest signal** for injury risk?
5. **What patterns emerge** across the 7-day lookback window?

Every finding in this notebook directly informs a downstream modeling decision —
from metric selection to class weighting strategy.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Ensure src/ is importable when running from notebooks/
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    ATHLETE_ID_COL,
    DATE_COL,
    DAY_FEATURES,
    INJURY_COL,
    N_DAY_BLOCKS,
    RANDOM_SEED,
    SENTINEL_VALUE,
)
from src.data_loading import get_feature_columns, load_day_data
from src.preprocessing.day_preprocessor import handle_sentinel_values
from src.utils.logging_config import setup_logging
from src.utils.plotting import INJURY_PALETTE, PALETTE, save_figure, set_style
from src.utils.reproducibility import set_global_seed

# --- Reproducibility & style ---
set_global_seed(RANDOM_SEED)
setup_logging()
set_style()

## 1. Data Loading and Shape Inspection

We load the raw day-approach CSV using our validated `load_day_data()` function,
which renames the original column headers (e.g., `"nr. sessions.1"`) into a clean
`day_{block}_{feature}` naming convention and validates the schema.

In [ ]:
df = load_day_data()
feature_cols = get_feature_columns(df)

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Feature columns: {len(feature_cols)} (7 blocks × 10 features)")
print(f"Metadata columns: {ATHLETE_ID_COL}, {INJURY_COL}, {DATE_COL}")
print(f"\nData types:\n{df.dtypes.value_counts().to_string()}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
df.head()

In [ ]:
df.describe()

### Data structure interpretation

The dataset uses a **rolling window design**:
- **day_0** = the current day (closest to the potential injury event)
- **day_6** = six days ago (most distant in the lookback window)

Each feature block captures training load at increasing temporal distance. This is
analogous to what a coach would review — the trailing week of training — to assess
cumulative load, fatigue, and recovery patterns.

The 10 features per day cover:
- **Volume**: `nr_sessions`, `total_km`
- **Intensity zones**: `km_z3_4` (threshold), `km_z5_t1_t2` (VO2max/anaerobic), `km_sprinting`
- **Cross-training**: `strength_training`, `hours_alternative`
- **Subjective measures**: `perceived_exertion`, `perceived_training_success`, `perceived_recovery`

This mix of objective (GPS/distance) and subjective (athlete self-report) data is
standard in modern sports science monitoring.

---

## 2. Target Distribution: The Imbalance Challenge

In [ ]:
target_counts = df[INJURY_COL].value_counts()
target_pct = df[INJURY_COL].value_counts(normalize=True) * 100

print("Target distribution:")
for label in [0, 1]:
    name = "No Injury" if label == 0 else "Injury"
    print(f"  {name} ({label}): {target_counts[label]:,} ({target_pct[label]:.2f}%)")
print(f"\nImbalance ratio: {target_counts[0] / target_counts[1]:.0f}:1")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))

labels = [f"No Injury — 0 ({target_pct[0]:.2f}%)", f"Injury — 1 ({target_pct[1]:.2f}%)"]
colors = [INJURY_PALETTE[0], INJURY_PALETTE[1]]
counts = [target_counts[0], target_counts[1]]

bars = ax.barh(labels, counts, color=colors, edgecolor="white", height=0.5)

for bar, count in zip(bars, counts):
    ax.text(
        bar.get_width() + max(counts) * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"n = {count:,}",
        va="center",
        fontweight="bold",
    )

ax.set_xlabel("Number of observations")
ax.set_title("Target Distribution — Day Approach", fontweight="bold")
ax.set_xlim(0, max(counts) * 1.15)
sns.despine(left=True)

save_figure(fig, "01_target_distribution", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Interpretation

The class imbalance is **extreme**: roughly a **72:1 ratio** of negative to positive
samples. This reflects reality — injury is a rare event even among elite athletes
who train at the physiological limit.

**Modeling implications:**
- **Accuracy is meaningless.** A naive model predicting "never injured" achieves ~98.6%
  accuracy while being completely useless. We will use **AUC-ROC** as the primary metric
  and **AUC-PR** (Precision-Recall) as a secondary metric that is more sensitive to
  the minority class.
- **Class weighting** (`class_weight='balanced'` or `scale_pos_weight`) will be our
  primary strategy to handle imbalance (ADR-003). This tells the model to penalize
  misclassified injuries more heavily, proportional to their rarity.
- **SMOTE** (Synthetic Minority Oversampling) will be tested as a secondary experiment,
  applied only within training folds to prevent data leakage.

---

## 3. Per-Athlete Analysis: Who Gets Injured?

Since the data comes from 74 individual athletes, understanding the **between-athlete
variability** is critical. Athletes differ in training volume, injury susceptibility,
and the length of their observation period.

In [ ]:
athlete_stats = (
    df.groupby(ATHLETE_ID_COL)
    .agg(
        n_observations=(INJURY_COL, "count"),
        n_injuries=(INJURY_COL, "sum"),
        injury_rate=(INJURY_COL, "mean"),
    )
    .sort_values("n_observations", ascending=False)
    .reset_index()
)

athlete_stats["injury_rate_pct"] = athlete_stats["injury_rate"] * 100

print(f"Number of unique athletes: {athlete_stats.shape[0]}")
print("\nObservations per athlete:")
print(f"  Mean:   {athlete_stats['n_observations'].mean():.0f}")
print(f"  Median: {athlete_stats['n_observations'].median():.0f}")
print(f"  Min:    {athlete_stats['n_observations'].min()}")
print(f"  Max:    {athlete_stats['n_observations'].max()}")
print(f"\nAthletes with zero injuries: {(athlete_stats['n_injuries'] == 0).sum()}")
print(f"Athletes with ≥1 injury:     {(athlete_stats['n_injuries'] > 0).sum()}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

colors = [
    INJURY_PALETTE[1] if n > 0 else INJURY_PALETTE[0]
    for n in athlete_stats["n_injuries"]
]

ax.bar(
    range(len(athlete_stats)),
    athlete_stats["n_observations"],
    color=colors,
    edgecolor="white",
    linewidth=0.5,
)

ax.set_xlabel("Athlete (sorted by number of observations)")
ax.set_ylabel("Number of observations")
ax.set_title(
    "Observations per Athlete (red = has injuries, blue = zero injuries)",
    fontweight="bold",
)
ax.set_xticks(range(0, len(athlete_stats), 5))
sns.despine()

save_figure(fig, "01_observations_per_athlete", subdir="eda", close=False)
plt.show()
plt.close(fig)

In [ ]:
athlete_sorted_by_rate = athlete_stats.sort_values("injury_rate_pct", ascending=True)
overall_mean = df[INJURY_COL].mean() * 100

fig, ax = plt.subplots(figsize=(10, 14))

colors = [
    INJURY_PALETTE[1] if r > 0 else PALETTE["neutral"]
    for r in athlete_sorted_by_rate["injury_rate_pct"]
]

ax.barh(
    range(len(athlete_sorted_by_rate)),
    athlete_sorted_by_rate["injury_rate_pct"],
    color=colors,
    edgecolor="white",
    height=0.7,
)

ax.axvline(overall_mean, color="black", linestyle="--", linewidth=1.2, label=f"Overall mean: {overall_mean:.2f}%")
ax.set_ylabel("Athlete")
ax.set_xlabel("Injury rate (%)")
ax.set_title("Injury Rate per Athlete", fontweight="bold")
ax.legend(loc="lower right")
ax.set_yticks(range(0, len(athlete_sorted_by_rate), 5))
sns.despine()

save_figure(fig, "01_injury_rate_per_athlete", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Interpretation

Athletes differ **dramatically** in both data volume and injury frequency:

- **Data volume heterogeneity**: Some athletes contribute thousands of observations
  (years of daily monitoring), while others have only a few hundred (a single season).
  This means some athletes dominate the training set.

- **Injury rate heterogeneity**: Some athletes have zero injuries across their entire
  observation period, while others have rates well above the 1.36% average. This likely
  reflects both individual injury susceptibility and differences in training intensity.

**Modeling implication — GroupKFold (ADR-006):**
We **must** split data by athlete, not by row. If the same athlete appears in both
training and test sets, the model could learn athlete-specific baselines (e.g., "athlete X
always has high exertion") rather than generalizable injury predictors. `GroupKFold`
ensures all observations from one athlete appear in exactly one fold.

---

## 4. Temporal Coverage: When Were Athletes Observed?

Understanding the temporal span of each athlete's data helps us assess whether
the dataset covers a representative period and whether injury events cluster
in specific time windows.

In [ ]:
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

temporal = (
    df.groupby(ATHLETE_ID_COL)
    .agg(
        first_date=(DATE_COL, "min"),
        last_date=(DATE_COL, "max"),
        n_obs=(INJURY_COL, "count"),
        injury_rate=(INJURY_COL, "mean"),
    )
    .sort_values("first_date")
    .reset_index()
)

temporal["span_days"] = (temporal["last_date"] - temporal["first_date"]).dt.days

print(f"Dataset time span: {temporal['first_date'].min()} → {temporal['last_date'].max()}")
print("\nObservation span per athlete:")
print(f"  Mean:   {temporal['span_days'].mean():.0f} days ({temporal['span_days'].mean() / 365:.1f} years)")
print(f"  Median: {temporal['span_days'].median():.0f} days")
print(f"  Min:    {temporal['span_days'].min()} days")
print(f"  Max:    {temporal['span_days'].max()} days")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))

norm = plt.Normalize(
    vmin=temporal["injury_rate"].min(),
    vmax=temporal["injury_rate"].max(),
)
cmap = plt.cm.RdYlBu_r

for i, row in temporal.iterrows():
    color = cmap(norm(row["injury_rate"]))
    ax.barh(
        i,
        (row["last_date"] - row["first_date"]).days,
        left=row["first_date"],
        color=color,
        edgecolor="white",
        height=0.7,
        linewidth=0.3,
    )

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, label="Injury rate", shrink=0.6)

ax.set_ylabel("Athlete (sorted by first observation date)")
ax.set_xlabel("Date")
ax.set_title("Temporal Coverage per Athlete (color = injury rate)", fontweight="bold")
ax.set_yticks(range(0, len(temporal), 5))
sns.despine()

save_figure(fig, "01_temporal_coverage", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Interpretation

Athletes enter and leave the dataset at different times — this is typical of
**longitudinal sport science studies** where athletes join programs, transfer clubs,
or retire at different points.

Key observations:
- **Unequal coverage**: some athletes have years of daily data, others only a few months.
  This affects statistical power: athletes with more data provide more stable estimates.
- **Staggered entry**: athletes don't all start at the same date, meaning the dataset
  captures different training and competition contexts.

This further justifies **athlete-level splitting** (GroupKFold) over temporal splitting
(TimeSeriesSplit). In classical time series, you split by date. But here, the strongest
structural grouping is by athlete — an athlete's training pattern is more self-correlated
than the overall population trend (ADR-006).

---

## 5. Feature Distributions: What Does a Training Day Look Like?

We focus on **day_0** (the current day) to understand the raw feature distributions
before examining temporal patterns across the 7-day window. Day 0 is the most recent
data point — the closest to the potential injury event.

In [ ]:
day0_cols = [f"day_0_{feat}" for feat in DAY_FEATURES]

fig, axes = plt.subplots(4, 3, figsize=(16, 16))
axes = axes.flatten()

for i, col in enumerate(day0_cols):
    ax = axes[i]
    data = df[col].dropna()
    ax.hist(data, bins=50, color=PALETTE["primary"], alpha=0.7, edgecolor="white")
    ax.set_title(col.replace("day_0_", ""), fontweight="bold", fontsize=11)
    ax.set_ylabel("Count")
    # Add median line
    median_val = data.median()
    ax.axvline(median_val, color=PALETTE["negative"], linestyle="--", linewidth=1,
               label=f"median={median_val:.2f}")
    ax.legend(fontsize=8)

# Hide unused subplots
for j in range(len(day0_cols), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Feature Distributions — Day 0 (Current Day)", fontweight="bold", fontsize=14, y=1.01)
fig.tight_layout()

save_figure(fig, "01_day0_feature_distributions", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Interpretation

The feature distributions reveal the **training behavior** of elite runners:

- **Volume features** (`nr_sessions`, `total_km`): Right-skewed with a large spike at
  zero. This makes sense — rest days (no training) are common even for elite athletes.
  On active days, most sessions are moderate distance.

- **Intensity features** (`km_z3_4`, `km_z5_t1_t2`, `km_sprinting`): **Heavily
  zero-inflated**. High-intensity work is done sparingly — a hallmark of polarized
  training, where ~80% of sessions are easy and only ~20% are hard. This is the
  dominant training paradigm in endurance sports.

- **Cross-training** (`strength_training`, `hours_alternative`): Binary-like or nearly
  zero-dominated. Strength work and alternative training (cycling, swimming, etc.) are
  supplementary activities.

- **Subjective features** (`perceived_exertion`, `perceived_training_success`,
  `perceived_recovery`): More continuous distributions, but note the presence of
  **sentinel values (−0.01)** that we will analyze in the next section.

**Modeling implication**: The heavy zero-inflation means **tree-based models** (Random
Forest, XGBoost) can naturally handle the zero/non-zero split as a binary decision at
each node. Linear models may struggle without explicit feature engineering.

---

## 6. Sentinel Value Analysis: Decoding Rest Days

The dataset uses **−0.01** as a sentinel value to indicate "no data recorded" —
typically meaning a rest day with no subjective rating. Understanding its prevalence
across features and temporal blocks is critical for preprocessing.

**Why not just use 0?** Because a zero value in training features has a real meaning
(e.g., 0 km run), while −0.01 explicitly signals "this data point was not collected."
For perceived metrics (exertion, recovery), if the athlete did not train, there is
no subjective rating to report.

In [ ]:
sentinel_pct = pd.DataFrame(
    index=range(N_DAY_BLOCKS),
    columns=DAY_FEATURES,
    dtype=float,
)

for block in range(N_DAY_BLOCKS):
    for feat in DAY_FEATURES:
        col = f"day_{block}_{feat}"
        sentinel_pct.loc[block, feat] = (df[col] == SENTINEL_VALUE).mean() * 100

print("Sentinel value (−0.01) prevalence (%) by feature and day block:\n")
print(sentinel_pct.round(2).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

sns.heatmap(
    sentinel_pct.astype(float),
    annot=True,
    fmt=".1f",
    cmap="Blues",
    cbar_kws={"label": "% of rows with sentinel (−0.01)"},
    linewidths=0.5,
    ax=ax,
)

ax.set_xlabel("Feature")
ax.set_ylabel("Day block")
ax.set_title("Sentinel Value (−0.01) Prevalence by Feature × Day Block", fontweight="bold")
ax.set_yticklabels([f"day_{i}" for i in range(N_DAY_BLOCKS)], rotation=0)
ax.set_xticklabels(DAY_FEATURES, rotation=45, ha="right")

save_figure(fig, "01_sentinel_heatmap", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Interpretation

The sentinel analysis confirms our expectations:

- **Sentinel values appear exclusively in perceived/subjective features**: `perceived_exertion`,
  `perceived_training_success`, and `perceived_recovery`. Volume and intensity features
  use real zeros for rest days.

- **Consistent across day blocks**: The sentinel percentage is roughly uniform across
  days 0–6, indicating that rest day patterns are stable within the 7-day window.

- **Prevalence**: A significant fraction of observations have sentinels, reflecting that
  athletes regularly take rest days (as expected in any training program).

**Preprocessing decision (ADR-007)**: We replace −0.01 → 0.0. The reasoning is
straightforward: if an athlete did not train on a given day, their perceived exertion
is zero (no effort), their perceived recovery is zero (no training to recover from),
and their perceived training success is zero (no training to rate). This is the
semantically correct value, and it prevents the artificial −0.01 from introducing
noise into distance-based and correlation-based analyses.

---

## 7. Correlation Structure: Redundancy and Signal

We analyze correlations at two levels:
1. **Within day 0** — which features co-vary on the same day?
2. **Across day blocks** — how correlated is the same feature across adjacent days?

**Important**: We first replace sentinel values to avoid distorted correlations.

In [ ]:
# Clean sentinel values before correlation analysis
df_clean = handle_sentinel_values(df)

# --- Within-day correlation (day 0 features) ---
day0_data = df_clean[[f"day_0_{feat}" for feat in DAY_FEATURES]]
day0_data.columns = DAY_FEATURES  # short names for readability
corr_day0 = day0_data.corr()

fig, ax = plt.subplots(figsize=(10, 8))

mask = np.triu(np.ones_like(corr_day0, dtype=bool), k=1)
sns.heatmap(
    corr_day0,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    ax=ax,
)

ax.set_title("Feature Correlation Matrix — Day 0", fontweight="bold")

save_figure(fig, "01_day0_correlation_matrix", subdir="eda", close=False)
plt.show()
plt.close(fig)

In [ ]:
# --- Cross-block correlation for total_km ---
total_km_cols = [f"day_{i}_total_km" for i in range(N_DAY_BLOCKS)]
cross_block = df_clean[total_km_cols].corr()
cross_block.index = [f"day_{i}" for i in range(N_DAY_BLOCKS)]
cross_block.columns = [f"day_{i}" for i in range(N_DAY_BLOCKS)]

fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(
    cross_block,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    ax=ax,
)

ax.set_title("Cross-Block Correlation — total_km (Day 0 to Day 6)", fontweight="bold")

save_figure(fig, "01_cross_block_correlation_total_km", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Interpretation

**Within-day correlations (day 0):**
- **Volume features cluster together**: `total_km`, `km_z3_4`, `km_z5_t1_t2` are
  positively correlated — a high-volume day tends to include more intensity work.
- **Subjective metrics form their own cluster**: `perceived_exertion`, `perceived_training_success`,
  and `perceived_recovery` are intercorrelated, reflecting that athlete self-reports
  tend to move together.
- **Cross-cluster links**: `perceived_exertion` correlates with volume features (harder
  sessions feel harder), while `perceived_recovery` may show weaker or inverse correlations.

**Cross-block correlations (total_km across days):**
- **Temporal autocorrelation decays with distance**: day 0 and day 1 are more correlated
  than day 0 and day 6. This reflects real training **periodization** — athletes
  structure their training in patterns (e.g., hard-easy-hard or block periodization),
  creating short-range dependencies.
- The 7-day window captures approximately one **microcycle** (a standard training block
  in periodization theory), which is why the lookback period was chosen.

**Modeling implication**: High cross-block correlation means some features carry
redundant information. Tree-based models handle this well (they select the best split
regardless), but for Logistic Regression, multicollinearity could inflate coefficient
variance.

---

## 8. Feature-Target Relationships: What Predicts Injury?

We compare the distribution of each day-0 feature between **injured** and
**non-injured** observations. Given the extreme imbalance (1.36% positive),
even small distribution shifts are meaningful.

We use violin plots with overlaid boxplots to show both the shape of the
distribution and summary statistics.

In [ ]:
n_injured = df_clean[df_clean[INJURY_COL] == 1].shape[0]
n_not_injured = df_clean[df_clean[INJURY_COL] == 0].shape[0]

# Build descriptive string labels → seaborn gets clean string categories,
# no np.int64 key issues, no set_xticklabels needed
label_no_injury = f"No Injury\n(n={n_not_injured:,})"
label_injury = f"Injury\n(n={n_injured:,})"
label_order = [label_no_injury, label_injury]
str_palette = {label_no_injury: INJURY_PALETTE[0], label_injury: INJURY_PALETTE[1]}

df_plot = df_clean.assign(
    injury_label=df_clean[INJURY_COL].map({0: label_no_injury, 1: label_injury})
)

fig, axes = plt.subplots(4, 3, figsize=(18, 18))
axes = axes.flatten()

for i, feat in enumerate(DAY_FEATURES):
    ax = axes[i]
    col = f"day_0_{feat}"

    sns.violinplot(
        data=df_plot,
        x="injury_label",
        y=col,
        hue="injury_label",
        order=label_order,
        hue_order=label_order,
        palette=str_palette,
        legend=False,
        ax=ax,
        inner="box",
        cut=0,
    )

    ax.set_title(feat, fontweight="bold", fontsize=11)
    ax.set_xlabel("")

# Hide unused subplots
for j in range(len(DAY_FEATURES), len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    "Feature Distributions by Injury Status — Day 0",
    fontweight="bold",
    fontsize=14,
    y=1.01,
)
fig.tight_layout()
save_figure(fig, "01_day0_features_by_injury", subdir="eda", close=False)
plt.show()

### Interpretation

The violin plots reveal the **feature-target relationships** that our models will
try to exploit:

- **Substantial overlap**: For most features, the distributions of injured and
  non-injured groups overlap heavily. This is expected — injury is a **multi-causal**
  phenomenon influenced by genetics, biomechanics, psychology, and training load
  simultaneously. No single feature cleanly separates the classes.

- **Subtle shifts matter**: Look for features where the median, spread, or tail
  differs between groups. For example:
  - **`perceived_recovery`**: If lower for injured athletes, it suggests under-recovery
    as a risk factor — a well-established finding in sports science.
  - **`total_km`** or intensity features: If slightly elevated for injured athletes,
    it supports the "too much too soon" hypothesis.

- **Small sample caveat**: The injury group contains far fewer samples, so its violin
  plot is noisier. We should not over-interpret apparent differences without
  statistical validation in the modeling phase.

**Why the prediction task is hard**: The paper's best AUC-ROC of 0.724 reflects
this overlap. Injury prediction from training load alone is inherently limited because
training load is only one piece of the puzzle.

---

## 9. The Rolling Window: A Coach's Perspective

Each row in the dataset represents a coach's view of the **last 7 days**. Day 0 is
today, day 6 is six days ago. By plotting how key features evolve across this window
— split by injury outcome — we can detect **pre-injury training patterns**.

This is the ML equivalent of what a sports scientist would call **acute:chronic
workload ratio (ACWR) analysis** — comparing recent training load to the baseline.

In [ ]:
key_features = ["total_km", "perceived_exertion", "perceived_recovery"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, feat in zip(axes, key_features):
    for label, color, name in [
        (0, INJURY_PALETTE[0], "No Injury"),
        (1, INJURY_PALETTE[1], "Injury"),
    ]:
        subset = df_clean[df_clean[INJURY_COL] == label]
        means = [subset[f"day_{d}_{feat}"].mean() for d in range(N_DAY_BLOCKS)]
        stds = [subset[f"day_{d}_{feat}"].std() for d in range(N_DAY_BLOCKS)]

        ax.plot(range(N_DAY_BLOCKS), means, color=color, marker="o", label=name, linewidth=2)
        ax.fill_between(
            range(N_DAY_BLOCKS),
            [m - s * 0.1 for m, s in zip(means, stds)],
            [m + s * 0.1 for m, s in zip(means, stds)],
            alpha=0.15,
            color=color,
        )

    ax.set_xlabel("Day block (0 = today, 6 = six days ago)")
    ax.set_ylabel(f"Mean {feat}")
    ax.set_title(feat.replace("_", " ").title(), fontweight="bold")
    ax.legend()
    ax.set_xticks(range(N_DAY_BLOCKS))
    ax.set_xticklabels([f"day_{i}" for i in range(N_DAY_BLOCKS)])
    sns.despine(ax=ax)

fig.suptitle(
    "Training Load Across the 7-Day Window by Injury Status",
    fontweight="bold",
    fontsize=14,
    y=1.03,
)
fig.tight_layout()

save_figure(fig, "01_rolling_window_by_injury", subdir="eda", close=False)
plt.show()
plt.close(fig)

### Interpretation

This visualization is the **domain expertise showcase** of the notebook. Here's how
to read it through the lens of sports science:

- **`total_km` pattern**: If injured athletes show elevated distance across the window
  (especially in days 3–6), it suggests **cumulative overload** — too many hard days
  without adequate recovery. In periodization theory, this is a failed microcycle where
  the athlete did not respect the hard-easy principle.

- **`perceived_exertion` pattern**: If exertion is consistently higher for injured
  athletes, it suggests they were training at a perceived intensity that exceeded their
  recovery capacity. This is the classic **overtraining signal** that coaches and sports
  scientists monitor daily.

- **`perceived_recovery` pattern**: If recovery is lower for injured athletes, it's
  the most clinically relevant finding — **under-recovery is a stronger injury
  predictor than overtraining** in modern sports science literature. An athlete who
  consistently reports low recovery is in a state of accumulated fatigue.

- **Flat vs. trending patterns**: If the lines are relatively flat across days 0–6,
  it means the 7-day window captures a steady-state rather than a dynamic change.
  If there's a trend (e.g., declining recovery from day 6 to day 0), it captures
  the *trajectory* of fatigue, not just its level.

**Connection to the acute:chronic workload ratio (ACWR):** The ACWR framework,
popularized by Blanch and Gabbett (2016), compares short-term (acute, ~1 week)
training load to long-term (chronic, ~4 weeks) load. An ACWR > 1.5 ("spike")
is associated with elevated injury risk. While this 7-day window doesn't directly
compute ACWR, the day-to-day progression across the window captures a similar
concept within a microcycle.

---

## 10. Key Insights and Implications for Modeling

### Summary of findings

1. **Extreme class imbalance (1.36% positive)**: Accuracy is meaningless as a metric.
   We must use **AUC-ROC** (primary) and **AUC-PR** (secondary). Class-weighted models
   are essential (ADR-003).

2. **Athlete heterogeneity**: Both data volume and injury rates vary dramatically
   across athletes. **GroupKFold by athlete ID** is mandatory to prevent the model
   from memorizing athlete-specific baselines rather than learning generalizable
   injury signals (ADR-006).

3. **Zero-inflated features**: Many training features (especially intensity zones
   and cross-training) are dominated by zeros. **Tree-based models** can naturally
   exploit the zero/non-zero boundary, while linear models would need feature
   engineering.

4. **Sentinel values encode rest days**: The −0.01 sentinel appears exclusively in
   perceived/subjective features on rest days. Replacement with 0.0 is semantically
   correct — "no training" means "zero perceived effort" (ADR-007).

5. **Temporal autocorrelation reflects periodization**: Training features are correlated
   across adjacent days, reflecting real periodization patterns. The 7-day window
   captures approximately one microcycle of training.

6. **Weak feature-target separation**: The distributions of injured and non-injured
   groups overlap substantially across all features. This explains the paper's moderate
   AUC-ROC of 0.724 — injury prediction from training load alone is a genuinely hard
   problem because training load is only one risk factor among many (genetics,
   biomechanics, psychology, nutrition, sleep).

### Next steps

→ **Notebook 02**: Explore the week-level approach, which trades temporal granularity
for richer weekly aggregations and introduces the relative km ratio features.

→ **Notebook 03**: Implement preprocessing (sentinel replacement, scaling, train/test
split by athlete) building on the insights from this EDA.